# Monte Carlo simulation

**Curriculum:** advanced quant track.

**Prerequisites:** work through `01_foundations/core_types_and_money.ipynb` and `01_foundations/math_toolkit.ipynb`. This notebook assumes comfort with `Money`-like results and basic probability.

**In this notebook:** Monte Carlo option pricing in `finstack_quant.models.monte_carlo` — analytical Black–Scholes benchmarks, canonical European, path-dependent, and American (LSMC) pricers, and an ATM European comparison.


## Monte Carlo option pricing

Under **risk-neutral** pricing, a European derivative value is the discounted expectation of its payoff under a model for the underlying. When no closed form exists, we **simulate** many paths of the process, average payoffs, and discount — the **Monte Carlo** estimator.

For a European call on a stock following geometric Brownian motion (GBM), the Black–Scholes formula gives the exact benchmark. Monte Carlo should agree within simulation error (stderr shrinks like $1/\sqrt{N}$ for $N$ paths).

`finstack_quant.models.monte_carlo` exposes:

- **Analytical** `models.bs_price` (`is_call=True/False`) for sanity checks.
- **High-level pricing** through `EuropeanPricer`.
- **Exotics:** `PathDependentPricer` (e.g. Asian), `LsmcPricer` (American via Longstaff–Schwartz).

Process and payoff parameters (rate, volatility, strike, etc.) are passed **directly** as numeric arguments to the pricer constructors and methods — there are no standalone Python parameter-object classes.


### Black–Scholes (analytical benchmark)

Use these closed-form prices to validate Monte Carlo output and to check **put–call parity**: $C - P = e^{-rT}(F - K)$ with forward $F = S e^{(r-q)T}$.


In [ ]:
from finstack_quant.models import bs_price
import math

from finstack_quant.models.monte_carlo import (
    EuropeanPricer,
    LsmcPricer,
    PathDependentPricer,
)

spot = strike = 100.0
r = 0.05
T = 1.0
bs_call = bs_price(spot, strike, r, 0.0, 0.20, T, True)
bs_put = bs_price(spot, strike, r, 0.0, 0.20, T, False)
disc = math.exp(-r * T)
print(f"BS Call: {bs_call:.6f}")
print(f"BS Put: {bs_put:.6f}")
print(f"Put-Call Parity: C-P = {bs_call - bs_put:.6f}, S-K*e^(-rT) = {spot - strike * disc:.6f}")


### `EuropeanPricer` (single-step-style European MC)

`EuropeanPricer` runs a straightforward GBM simulation and returns a `MoneyEstimate`: mean estimate in `mean` (a `Money`-like amount + currency), standard error, asymptotic confidence band, and path count.


In [ ]:
pricer = EuropeanPricer(num_paths=50_000, seed=42)
result = pricer.price_call(spot=100.0, strike=100.0, rate=0.05, div_yield=0.0, vol=0.20, expiry=1.0)
print(f"MC Call: {result.mean.amount:.6f}")
print(f"Currency: {result.mean.currency.code}")
print(f"Std error: {result.stderr:.6f}")
print(f"95% CI: [{result.ci_lower.amount:.6f}, {result.ci_upper.amount:.6f}]")
print(f"Num paths: {result.num_paths}")

result_put = pricer.price_put(spot=100.0, strike=100.0, rate=0.05, div_yield=0.0, vol=0.20, expiry=1.0)
print(f"MC Put: {result_put.mean.amount:.6f}")


### Stochastic processes

The Python pricers fix the underlying dynamics to **geometric Brownian motion (GBM)** and accept risk-neutral drift parameters (``rate``, ``div_yield``, ``vol``) directly as numeric arguments. Alternative processes (Heston, CIR, Schwartz–Smith, …) are implemented in the Rust crate `finstack-quant-models` and can be consumed from Rust or via custom extensions; they are not surfaced as Python classes today.

A few process-level helpers *are* bound as functions — for example `heston_satisfies_feller`, which validates Heston parameters and tests the **Feller condition** $2\kappa\theta > \xi^2$. When that condition holds, the variance process stays strictly positive; when it fails, discretization schemes must handle variance hitting zero. See `monte_carlo/stochastic_processes.ipynb` for the full process tour.


In [ ]:
from finstack_quant.models.monte_carlo import heston_satisfies_feller

gbm_params = dict(rate=0.05, div_yield=0.0, vol=0.20)
print(f"GBM params: {gbm_params}")

heston_params = dict(
    rate=0.05, div_yield=0.0, v0=0.04,
    kappa=2.0, theta=0.04, xi=0.3, rho=-0.7,
)
# The Feller condition 2*kappa*theta > xi^2 keeps the Heston variance process
# strictly positive. Ask the binding rather than re-deriving it: it validates the
# parameters (positive kappa/theta/vol-of-vol) as well as testing the inequality.
feller_ok = heston_satisfies_feller(
    heston_params["kappa"], heston_params["theta"], heston_params["xi"],
)
print(f"Heston params: {heston_params}")
print(f"Feller satisfied (2*kappa*theta > xi^2): {feller_ok}")

### Path-dependent: arithmetic Asian call

`PathDependentPricer` prices payoffs that depend on the whole path; here an **arithmetic average** Asian call with fixings at each simulated step.


In [ ]:
asian_pricer = PathDependentPricer(num_paths=10_000, seed=42)
asian_result = asian_pricer.price_asian_call(
    spot=100.0, strike=100.0, rate=0.05, div_yield=0.0,
    vol=0.20, expiry=1.0, num_steps=252,
)
print(f"Asian Call: {asian_result.mean.amount:.6f}")


### American options: `LsmcPricer`

Longstaff–Schwartz fits an exercise policy on a finite Bermudan grid. The exact optimal American value dominates the European value, but that economic statement is not a deterministic inequality for noisy fitted-policy estimates.

Use `price_american_put_unbiased` with a distinct pricing seed to separate policy training from valuation. It removes the reuse of training paths; it does not remove grid or policy approximation error. Report estimator and simulated-path counts: this API's antithetic setting counts **pairs**, unlike a plain European run.

In [ ]:
from finstack_quant.models import bs_price

lsmc = LsmcPricer(num_paths=5000, seed=42, num_steps=64, antithetic=True)
am_put = lsmc.price_american_put_unbiased(
    100.0, 100.0, 0.05, 0.0, 0.30, 1.0, pricing_seed=1042
)
bs_put_ref = bs_price(100.0, 100.0, 0.05, 0.0, 0.30, 1.0, False)
print(f"Held-out Bermudan policy value: {am_put.mean.amount:.6f}; SE={am_put.stderr:.6f}")
print(f"European analytical put:      {bs_put_ref:.6f}")
print(f"Policy-value gap to European: {am_put.mean.amount - bs_put_ref:.6f}")
print(f"Pricing estimators={am_put.num_paths}; simulated pricing paths={am_put.num_simulated_paths}")
print("The independent training pass costs another path set of the same size.")
assert am_put.num_paths == 5000 and am_put.num_simulated_paths == 10_000

## Mini-example: one ATM call, two estimators

Compare the Black–Scholes analytical result with `EuropeanPricer` using 50,000 paths. The table reports price, error versus Black–Scholes, and the Monte Carlo confidence interval.


In [ ]:
from finstack_quant.models import bs_price
spot = strike = 100.0
rate = 0.05
div_yield = 0.0
vol = 0.20
expiry = 1.0
num_paths = 50_000
seed = 42

bs = bs_price(spot, strike, rate, div_yield, vol, expiry, True)

ep = EuropeanPricer(num_paths=num_paths, seed=seed)
r_ep = ep.price_call(
    spot=spot, strike=strike, rate=rate, div_yield=div_yield, vol=vol, expiry=expiry,
)


def fmt_ci(r):
    return f"[{r.ci_lower.amount:.6f}, {r.ci_upper.amount:.6f}]"

rows = [
    ("Black-Scholes (exact)", bs, 0.0, "n/a (exact)"),
    ("EuropeanPricer (50k)", r_ep.mean.amount, r_ep.mean.amount - bs, fmt_ci(r_ep)),
]

print(f"Parameters: S={spot}, K={strike}, r={rate}, q={div_yield}, sigma={vol}, T={expiry}y")
print()
w0, w1, w2, w3 = 28, 14, 14, 36
print(f"{'Method':<{w0}} {'Price':>{w1}} {'Err vs BS':>{w2}} {'95% CI':<{w3}}")
print("-" * (w0 + w1 + w2 + w3))
for name, px, err, ci in rows:
    print(f"{name:<{w0}} {px:>{w1}.6f} {err:>{w2}.6f} {ci:<{w3}}")


## Antithetic arithmetic: compare equal simulated-path budgets

Use `LsmcPricer(num_steps=1)` for a maturity-only ATM put, so no early-exercise regression or time-zero exercise value obscures this experiment. With antithetics enabled, $N$ means **N independent pair averages and 2N simulated paths**. Compare with 2N plain paths.

For exact one-step GBM, $S_+(T)S_-(T)=[S_0e^{(r-q-\sigma^2/2)T}]^2$. Reconstruct each reflected terminal spot from a plain captured primary path, discount both put payoffs, average the pair, then calculate $s_{\mathrm{pairs}}/\sqrt{N}$. This independently checks the pricer's reported standard error.

In [ ]:
from finstack_quant.models.monte_carlo import LsmcPricer
import statistics
from finstack_quant.models.monte_carlo import simulate_gbm_paths

pairs = 2048
seed = 17
discount = math.exp(-0.05)
captured = simulate_gbm_paths(100, 0.05, 0.0, 0.20, 1.0, 1, 2 * pairs, seed=seed)
terminals = [path[-1] for path in captured.paths]
plain_payoffs = [discount * max(100 - terminal, 0) for terminal in terminals]
reflection_product = (100 * math.exp(0.05 - 0.5 * 0.20**2)) ** 2
pair_payoffs = [
    0.5 * (plain_payoffs[i] + discount * max(100 - reflection_product / terminal, 0))
    for i, terminal in enumerate(terminals[:pairs])
]
print("method | estimators | simulated paths | mean | reported SE | manually calculated SE")
results = []
for antithetic, count, payoffs in [(True, pairs, pair_payoffs), (False, 2 * pairs, plain_payoffs)]:
    estimate = LsmcPricer(count, seed, False, 1, antithetic=antithetic).price_american_put(
        100, 100, 0.05, 0, 0.20, 1.0
    )
    manual_mean = statistics.mean(payoffs)
    manual_se = statistics.stdev(payoffs) / math.sqrt(count)
    assert math.isclose(estimate.mean.amount, manual_mean, abs_tol=1e-12)
    assert math.isclose(estimate.stderr, manual_se, abs_tol=1e-12)
    assert estimate.num_paths == count and estimate.num_simulated_paths == 2 * pairs
    print(f"{'antithetic' if antithetic else 'plain':10s} | {count:10d} | "
          f"{estimate.num_simulated_paths:15d} | {manual_mean:.6f} | "
          f"{estimate.stderr:.6f} | {manual_se:.6f}")
    results.append(estimate)
print(f"Equal-path-budget variance ratio (plain/paired): {(results[1].stderr/results[0].stderr)**2:.4f}")
assert results[0].stderr < results[1].stderr  # This fixed monotone payoff, not a universal guarantee.

## Takeaways

- Standard error scales as $1/\sqrt{N}$ for independent finite-variance estimators. A single realized error need not shrink monotonically.
- A nominal 95% Monte Carlo interval has approximate repeated-sampling coverage, not guaranteed coverage for every seed.
- A one-step European under exact GBM isolates sampling error from discretization error.
- Count independent estimators and raw simulated paths separately when comparing variance reduction.
- Match averaging schedules and monitoring conventions before comparing exotic-option models.
- LSMC held-out pricing addresses training-sample reuse, not all policy or grid error. Use independent pricing seeds and inspect uncertainty.

### Named Monte Carlo estimate types

Currency-valued pricers return `MoneyEstimate`; scalar Monte Carlo routines expose the sibling `Estimate` type for non-money outputs.

In [ ]:
from finstack_quant.models.monte_carlo import Estimate, MoneyEstimate

print("MoneyEstimate type:", MoneyEstimate.__name__)
print("call result typed:", isinstance(result, MoneyEstimate))
print("Scalar estimate type available:", Estimate.__name__)